# 읽는 분께 — 이 노트북의 구성

AI 시스템반도체 설계 2기 · 가위바위보 On-Device 프로젝트 · 정보윤 · 2026-09-18

**교수님이 주신 원본 학습 노트북에, 이번 프로젝트에서 직접 추가한 셀이 섞여 있습니다.**
보고서에 실은 수치가 어디서 나왔는지를 남기기 위해 지우지 않았습니다.

## 원본 셀 (교수님 제공)

`g-drive 마운트` → `경로 설정` → `RPS 데이터셋 준비` → `Ultralytics 설치` →
`학습` → `LiteRT로 변환` → `tflite 파일 저장`

판을 새로 학습할 때마다 경로 셀의 `dataset_zip` / `dataset_root` 와
저장 셀의 `OUT` 만 바꿔 가며 v6 · v7 · v9 · v10 을 만들었습니다.

## 직접 추가한 셀과 보고서 대응

| 추가한 것 | 한 일 | 보고서 |
|---|---|---|
| 고정 시험지 채점 | 직접 촬영·라벨링한 233장(박스 328개)으로 v1 · v3 · v9 를 같은 잣대로 채점 | 4-1 두 시험지 비교표 |
| 새 사진 검출률 측정 | v3 로 새 사진을 검출해 "두 손 다 잡은 비율 32%" 확인 | 4-1 실사용 성능 |
| A/B 실험 | 사진은 같고 팀원 촬영분 88장의 라벨만 바꾼 두 판을 학습·채점 | 4-5 라벨 품질 가설 **기각** |
| 시드 반복 학습 | 같은 데이터로 시드만 0·1·2 로 바꿔 3회 학습 | 4-1 실행 간 변동폭 **0.142** |
| 신뢰도 문턱값 훑기 | 게임의 판정 절차를 그대로 재현해 문턱값별 정확도 측정 | 4-6 판정률 3.1배 |
| `args.yaml` 대조 | 학습 폴더가 어느 데이터셋으로 몇 epoch 돌았는지 확인 | 4-2 **2 epoch 중단 사고** 발견 |
| BEST 선택 방어 코드 | 폴더 이름이 아니라 `args.yaml` 의 `data:` 로 판을 식별하고, `epoch < 80` 이면 `RuntimeError` | 같은 사고의 재발 방지 |

## 남겨 둔 실패 기록

- GPU 사용량 한도로 멈춘 셀, `device=0` 을 CPU 런타임에서 실행해 난 오류가 그대로 있습니다.
- 정리하면 깔끔하지만, **무엇을 시도했고 왜 멈췄는지**가 사라집니다. 과정을 그대로 두는 편이 낫다고 판단했습니다.

## 함께 제출한 파일

`제출용_소스코드/README.md` 에 보드용 · 데이터셋 도구 17개 파일의 설명이 있습니다.


# RPS PreTrained YOLO Fine-tuning

In [ ]:
!pip -q install ultralytics "ai-edge-litert>=2.1.4"

from google.colab import drive
drive.mount('/content/drive')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 84.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.3/21.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 5.7 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
import glob, os
from ultralytics import YOLO

!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_real.zip -d /content
DATA = '/content/RPS_TestSet_real/data.yaml'
S = '/content/drive/MyDrive/files/save/'

MODELS = [
    ('v1  (강사님 104장)',      S + 'v1_best.tflite'),
    ('v3  (724장)',             S + 'v3/best.tflite'),
    ('v4  (1071장)',            S + 'v4/best.tflite'),
    ('v3 INT8 (보드에서 쓰는 것)', S + 'v3/best_int8.tflite'),
]

rows = []
for name, p in MODELS:
    if not os.path.exists(p):
        print('없음:', p); continue
    r = YOLO(p).val(data=DATA, imgsz=320, verbose=False, plots=False)
    ap = {c: r.box.maps[i] for i, c in enumerate(['scissors','rock','paper'])}
    rows.append((name, r.box.map50, r.box.map, r.box.mp, r.box.mr, ap))

print()
print('%-26s %7s %9s %7s %7s   %s' % ('모델','mAP50','mAP50-95','정밀도','재현율','클래스별 mAP50-95'))
print('-' * 104)
for n, m50, m, p_, r_, ap in rows:
    print('%-26s %7.3f %9.3f %7.3f %7.3f   가위 %.3f  주먹 %.3f  보 %.3f'
          % (n, m50, m, p_, r_, ap['scissors'], ap['rock'], ap['paper']))

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
Ultralytics 8.4.155 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/v1_best.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1622.7±910.8 MB/s, size: 53.0 KB)
val: Scanning /content/RPS_TestSet_real/labels/val... 135 images, 15 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 135/135 1.5Kit/s 0.1s
val: New cache created: /content/RPS_TestSet_real/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 135/135 32.0it/s 4.2s
                   all        135        240      0.473      0.598      0.526   

In [ ]:
import os, glob, shutil

# v3 폴더에 .pt 가 있으면 그게 제일 정확하고 빠르다. 없으면 float32 tflite.
pt = glob.glob(save_dir + 'v3/*.pt')
MODEL = pt[0] if pt else save_dir + 'v3/best.tflite'
print('사용할 모델:', MODEL)

!pip -q install ultralytics
!unzip -q -o /content/drive/MyDrive/files/captures.zip -d /content/cap

from ultralytics import YOLO
ROOT = '/content/cap/captures'
m = YOLO(MODEL)

total_jpg = total_txt = 0
for d in sorted(os.listdir(ROOT)):da
    p = os.path.join(ROOT, d)
    if not os.path.isdir(p) or d == '_bg':
        continue
    m.predict(source=p, imgsz=320, conf=0.35,
              save_txt=True, save=False,
              project='/content/auto', name=d, exist_ok=True, verbose=False)
    got = glob.glob('/content/auto/%s/labels/*.txt' % d)
    for t in got:
        shutil.copy(t, p)
    n_jpg = len(glob.glob(os.path.join(p, '*.jpg')))
    total_jpg += n_jpg; total_txt += len(got)
    print('%-20s 사진 %2d장 → 라벨 %2d개  (검출 실패 %d장)'
          % (d, n_jpg, len(got), n_jpg - len(got)))

print('-' * 52)
print('합계  사진 %d장  라벨 %d개  실패 %d장' % (total_jpg, total_txt, total_jpg - total_txt))

!cd /content/cap && zip -qr /content/drive/MyDrive/files/captures_labeled.zip captures
print('완료 — 드라이브 files/captures_labeled.zip 을 받으세요')

사용할 모델: /content/drive/MyDrive/files/save/v3/best.tflite
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.5/46.5 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 9.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
requirements: Ultralytics requirement ['ai-edge-litert>=2.1.4'] not found, attempting AutoUpdate...
Using Python 3.13.15 environment at: /usr
Resolved 8 packages in 249ms
Prepared 2 packages in 302ms
Installed 2 packages in 4ms
 + ai-edge-litert==2.2.0
 + backports-strenum==1.3.1

requirements: AutoUpdate success ✅ 0.9s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

Loading

In [ ]:
import os, glob, shutil
from google.colab import drive
drive.mount('/content/drive')

save_dir = '/content/drive/MyDrive/files/save/'

# 모델 찾기 — 어떤 이름으로 저장했는지 확인부터
cands = glob.glob(save_dir + '**/*.tflite', recursive=True)
for c in cands:
    print(c)

Mounted at /content/drive
/content/drive/MyDrive/files/save/RPS_MobileNetV2.tflite
/content/drive/MyDrive/files/save/RPS_MobileNetV2_Augmentation.tflite
/content/drive/MyDrive/files/save/mnist_pruning_whole_model.tflite
/content/drive/MyDrive/files/save/mnist_baseline_model.tflite
/content/drive/MyDrive/files/save/mnist_pruning_some_layer.tflite
/content/drive/MyDrive/files/save/mnist_ptq_dr.tflite
/content/drive/MyDrive/files/save/RPS_MobileNetV2_Augmentation_PTQ_INT8.tflite
/content/drive/MyDrive/files/save/RPS_MobileNetV2_Augmentation_QAT.tflite
/content/drive/MyDrive/files/save/v1_best.tflite
/content/drive/MyDrive/files/save/v1_best_int8.tflite
/content/drive/MyDrive/files/save/v1_best_w8a32.tflite
/content/drive/MyDrive/files/save/best_int8.tflite
/content/drive/MyDrive/files/save/best_w8a32.tflite
/content/drive/MyDrive/files/save/best.tflite
/content/drive/MyDrive/files/save/rps_yolo11n/weights/best_w8a32.tflite
/content/drive/MyDrive/files/save/rps_yolo11n/weights/best_int8.tf

## g-drive 마운트

In [2]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('g-drive mounted.')
    colab=True
except:
    print('local drive.')
    colab =False

Mounted at /content/drive
g-drive mounted.


## 경로 설정

In [ ]:
# 다른 경로에 저장하려면, 아래 경로를 수정하세요
if colab :
  save_dir = '/content/drive/MyDrive/files/save/'
  dataset_zip  = '/content/drive/MyDrive/files/RPS_Dataset_YOLO_v9.zip'
  dataset_root = '/content/RPS_Dataset_YOLO_v9'
  project_root = '/content/rps_runs'
else :
  save_dir = '../files/save/'
  dataset_zip = '../files/RPS_Dataset_YOLO.zip'
  dataset_root = './RPS_Dataset_YOLO'
  project_root = './rps_runs'

## RPS 데이터셋 준비

In [ ]:
!unzip -q -o {dataset_zip}

## Ultralytics 설치 및 import

In [3]:
!pip -q install -U ultralytics==8.4.98

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 22.3 MB/s eta 0:00:00


## 학습

In [ ]:
DATA = f'{dataset_root}/data.yaml'

!yolo detect train model=yolo11n.pt data={DATA} \
  imgsz=320 epochs=80 batch=32 device=0 workers=4 \
  project={project_root} degrees=15.0 flipud=0.5 \
  fliplr=0.5 mosaic=1.0 name=rps_yolo11n

New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO_v9/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic

## LiteRT로 변환

In [ ]:
import glob, os, io

WANT = 'RPS_Dataset_YOLO_v9'      # 이번에 학습한 데이터셋 이름

hit = []
for d in glob.glob(f'{project_root}/rps_yolo11n*'):
    a, csv, w = d + '/args.yaml', d + '/results.csv', d + '/weights/best.pt'
    if not (os.path.exists(a) and os.path.exists(csv) and os.path.exists(w)):
        continue
    data = ''
    for ln in io.open(a, encoding='utf-8'):
        if ln.startswith('data:'):
            data = ln.split(':', 1)[1].strip()
    ep = sum(1 for _ in open(csv)) - 1
    print('  후보 %-28s 데이터=%-38s %d에포크' % (os.path.basename(d), data, ep))
    if WANT in data:
        hit.append((ep, d))

if len(hit) != 1:
    raise RuntimeError('%s 로 학습한 폴더가 %d개다. 확인할 것.' % (WANT, len(hit)))
EPOCHS, RUN = hit[0]
BEST = os.path.join(RUN, 'weights', 'best.pt')
print('쓸 모델:', BEST, '| 학습 epoch:', EPOCHS)
if EPOCHS < 80:
    raise RuntimeError('학습이 %d epoch에서 끊겼다. 학습 셀부터 다시 실행할 것.' % EPOCHS)

# 양자화 안함
!yolo export model={BEST} format=litert imgsz=320 data={DATA}

  후보 rps_yolo11n-2                데이터=/content/RPS_Dataset_YOLO_v7/data.yaml 80에포크
  후보 rps_yolo11n                  데이터=/content/RPS_Dataset_YOLO_v6/data.yaml 80에포크
  후보 rps_yolo11n-3                데이터=/content/RPS_Dataset_YOLO_v9/data.yaml 80에포크
쓸 모델: /content/rps_runs/rps_yolo11n-3/weights/best.pt | 학습 epoch: 80
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n-3/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packag

In [ ]:
# PTQ(INT8) (입력과 출력은 float32)
!yolo export model={BEST} format=litert imgsz=320 data={DATA} quantize=8

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
WARNING ⚠️ LiteRT INT8 export does not support end2end models, disabling end2end branch.
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n-2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
LiteRT: collecting INT8 calibration images from 'data=/content/RPS_Dataset_YOLO_v7/data.yaml'
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1634.0±290.9 MB/s, size: 70.4 KB)
val: Scanning /content/RPS_Dataset_YOLO_v7/labels/test.cache... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 3.1Mit/s 0.0s
WARNING ⚠️ LiteRT: >300 images recommended for INT8 calibration, found 22 images.
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
F

In [ ]:
# PTQ(Dynamic Range)
!yolo export model={BEST} format=litert imgsz=320 data={DATA} quantize=w8a32

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs

PyTorch: starting from '/content/rps_runs/rps_yolo11n-2/weights/best.pt' with input shape (1, 3, 320, 320) BCHW and output shape(s) (1, 7, 2100) (5.2 MB)
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so

LiteRT: starting export with litert_torch 0.9.4...
(00:00) [START] LiteRT-Torch Convert
(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default
(00:01) [START] LiteRT-Torch Convert > Torch Export: serving_default > 
ExportedProgram Run Decompositions
/

## tflite 파일 저장

In [ ]:
OUT = save_dir + 'v9/'
!mkdir -p {OUT}
!cp {RUN}/weights/*.tflite {OUT}
!ls -lh {OUT}

total 11M
-rw------- 1 root root 11M Sep 18 04:11 best.tflite


In [ ]:
!cp -r {RUN} {save_dir}

In [ ]:
import pandas as pd
d = RUN
df = pd.read_csv(f'{d}/results.csv'); df.columns = df.columns.str.strip()
r = df.iloc[-1]
print('=== 강사님 시험지 22장 기준 ===')
print('epoch    %d' % r['epoch'])
print('mAP50    %.4f' % r['metrics/mAP50(B)'])
print('mAP50-95 %.4f' % r['metrics/mAP50-95(B)'])
print('P %.4f   R %.4f' % (r['metrics/precision(B)'], r['metrics/recall(B)']))

=== 강사님 시험지 22장 기준 ===
epoch    2
mAP50    0.7001
mAP50-95 0.2619
P 0.4809   R 0.3380


In [ ]:
!yolo val model={BEST} data={DATA} split=test

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1619.0±466.0 MB/s, size: 58.8 KB)
val: Scanning /content/RPS_Dataset_YOLO_v4/labels/test_hard.cache... 247 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 247/247 45.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 8.0it/s 2.0s
                   all        247        308      0.537      0.386      0.497       0.19
              scissors         83         94      0.318      0.223      0.254      0.106
                  rock        104        117          1     0.0777      0.659      0.219
                 paper         89         97      0.292      0.856      0.579      0.247
Speed: 0.4ms preprocess, 2.0ms inference, 0.0ms loss, 2.0ms postprocess per image
Results saved to /content/runs/detect

In [ ]:
!yolo val model={BEST} data={DATA} split=test imgsz=320

Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1846.8±337.2 MB/s, size: 67.0 KB)
val: Scanning /content/RPS_Dataset_YOLO_v4/labels/test_hard.cache... 247 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 247/247 47.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 7.8it/s 2.0s
                   all        247        308      0.537      0.386      0.497       0.19
              scissors         83         94      0.318      0.223      0.254      0.106
                  rock        104        117          1     0.0777      0.659      0.219
                 paper         89         97      0.292      0.856      0.579      0.247
Speed: 0.3ms preprocess, 2.2ms inference, 0.0ms loss, 2.6ms postprocess per image
Results saved to /content/runs/detect

In [ ]:
!yolo val model={save_dir}v3/best.tflite data={DATA} split=test imgsz=320

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/v3/best.tflite for LiteRT inference...
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1563.0±716.9 MB/s, size: 65.5 KB)
val: Scanning /content/RPS_Dataset_YOLO_v4/labels/test_hard.cache... 247 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 247/247 37.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 247/247 32.5it/s 7.6s
                   all        247        308      0.832      0.594      0.721      0.407
              scissors         83         94      0.886      0.413      0.641      0.349
    

In [ ]:
print('=============== v4 (1071장 학습) ===============')
!yolo val model={BEST} data={DATA} split=test imgsz=320

print('=============== v3 (724장, 비교용) ===============')
!yolo val model={save_dir}v3/best.tflite data={DATA} split=test imgsz=320

=============== v4 (1071장 학습) ===============
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11n summary (fused): 101 layers, 2,582,737 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1976.3±814.7 MB/s, size: 74.7 KB)
val: Scanning /content/RPS_Dataset_YOLO_v4/labels/test_hard.cache... 247 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 247/247 49.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 16/16 7.8it/s 2.1s
                   all        247        308      0.537      0.386      0.497       0.19
              scissors         83         94      0.318      0.223      0.254      0.106
                  rock        104        117          1     0.0777      0.659      0.219
                 paper         89         97      0.292      0.856      0.579      0.247
Speed: 0.4ms preprocess, 2.0ms inference, 0.0ms loss, 2.3ms postprocess p

In [ ]:
import glob, os
from ultralytics import YOLO

m = YOLO('/content/drive/MyDrive/files/save/v3/best_int8.tflite')
IMGS = [p for p in sorted(glob.glob('/content/RPS_TestSet_real/images/val/*.jpg'))
        if not os.path.basename(p).startswith('_bg')]
BGS  = [p for p in sorted(glob.glob('/content/RPS_TestSet_real/images/val/*.jpg'))
        if os.path.basename(p).startswith('_bg')]

def confs(paths):
    out = []
    for p in paths:                      # tflite 는 batch=1 고정이라 한 장씩
        r = m.predict(p, imgsz=320, conf=0.01, max_det=10, verbose=False)[0]
        out.append(sorted(float(x) for x in r.boxes.conf))
    return out

hand = confs(IMGS)
bg   = confs(BGS)

print('%6s %9s %9s %9s %12s' % ('문턱값','두손다','한손만','못잡음','배경오검출'))
print('-' * 52)
for c in (0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10):
    n2 = sum(1 for v in hand if sum(1 for x in v if x >= c) >= 2)
    n1 = sum(1 for v in hand if sum(1 for x in v if x >= c) == 1)
    n0 = len(hand) - n2 - n1
    fp = sum(sum(1 for x in v if x >= c) for v in bg)
    mark = '   <- 지금 게임 설정' if abs(c - 0.40) < 1e-6 else ''
    print('%6.2f %7d장 %7d장 %7d장 %9d개%s' % (c, n2, n1, n0, fp, mark))

Loading /content/drive/MyDrive/files/save/v3/best_int8.tflite for LiteRT inference...
   문턱값       두손다       한손만       못잡음        배경오검출
----------------------------------------------------
  0.40      31장      56장      33장         0개   <- 지금 게임 설정
  0.35      37장      60장      23장         0개
  0.30      43장      60장      17장         0개
  0.25      53장      55장      12장         0개
  0.20      67장      46장       7장         0개
  0.15      80장      37장       3장         0개
  0.10      97장      21장       2장         0개


In [ ]:
import glob, os, io
import numpy as np
from ultralytics import YOLO

m = YOLO('/content/drive/MyDrive/files/save/v3/best_int8.tflite')
D = '/content/RPS_TestSet_real'
IMGS = [p for p in sorted(glob.glob(D + '/images/val/*.jpg'))
        if not os.path.basename(p).startswith('_bg')]

# 정답: 왼쪽 손 클래스, 오른쪽 손 클래스
truth = {}
for p in IMGS:
    t = D + '/labels/val/' + os.path.basename(p)[:-4] + '.txt'
    rows = [l.split() for l in io.open(t) if len(l.split()) == 5]
    rows.sort(key=lambda q: float(q[1]))          # cx 순 = 왼쪽부터
    truth[p] = [int(q[0]) for q in rows]

# 한 번만 추론해서 (클래스, cx, 신뢰도) 를 모아 둔다
raw = {}
for p in IMGS:
    r = m.predict(p, imgsz=320, conf=0.01, max_det=10, verbose=False)[0]
    b = r.boxes
    raw[p] = [(int(c), float((x[0] + x[2]) / 2), float(s))
              for c, x, s in zip(b.cls, b.xyxy, b.conf)]

print('%6s %9s %9s %9s %10s' % ('문턱값', '판정정답', '판정오답', '박스부족', '정답률'))
print('-' * 52)
for c in (0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10, 0.05):
    ok = ng = few = 0
    for p in IMGS:
        d = [v for v in raw[p] if v[2] >= c]
        if len(d) < 2:
            few += 1; continue
        d = sorted(d, key=lambda v: -v[2])[:2]     # 신뢰도 높은 2개 (게임과 동일)
        d = sorted(d, key=lambda v: v[1])          # 왼쪽부터
        ok += ([v[0] for v in d] == truth[p])
        ng += ([v[0] for v in d] != truth[p])
    print('%6.2f %7d장 %7d장 %7d장 %9.1f%%'
          % (c, ok, ng, few, ok * 100.0 / len(IMGS)))

Loading /content/drive/MyDrive/files/save/v3/best_int8.tflite for LiteRT inference...
   문턱값      판정정답      판정오답      박스부족        정답률
----------------------------------------------------
  0.40      28장       3장      89장      23.3%
  0.35      30장       7장      83장      25.0%
  0.30      34장       9장      77장      28.3%
  0.25      42장      11장      67장      35.0%
  0.20      54장      13장      53장      45.0%
  0.15      64장      16장      40장      53.3%
  0.10      76장      21장      23장      63.3%
  0.05      84장      32장       4장      70.0%


In [ ]:
import os
from ultralytics import YOLO

!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_hold40.zip -d /content
HOLD = '/content/RPS_TestSet_hold40/data.yaml'
S = '/content/drive/MyDrive/files/save/'

MODELS = [
    ('v1  (104장)',        S + 'v1_best.tflite'),
    ('v3  (724장)',        S + 'v3/best.tflite'),
    ('v4  (1071장)',       S + 'v4/best.tflite'),
    ('v6  (971장)',        S + 'v6/best.tflite'),
    ('v6 INT8 (보드용)',   S + 'v6/best_int8.tflite'),
]

print()
print('%-20s %7s %9s %7s %7s   %s'
      % ('모델', 'mAP50', 'mAP50-95', '정밀도', '재현율', '클래스별 mAP50-95'))
print('-' * 100)
for name, p in MODELS:
    if not os.path.exists(p):
        print('없음:', p); continue
    r = YOLO(p).val(data=HOLD, imgsz=320, verbose=False, plots=False)
    a = r.box.maps
    print('%-20s %7.3f %9.3f %7.3f %7.3f   가위 %.3f  주먹 %.3f  보 %.3f'
          % (name, r.box.map50, r.box.map, r.box.mp, r.box.mr, a[0], a[1], a[2]))


모델                     mAP50  mAP50-95     정밀도     재현율   클래스별 mAP50-95
----------------------------------------------------------------------------------------------------
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/v1_best.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1766.6±777.2 MB/s, size: 51.2 KB)
val: Scanning /content/RPS_TestSet_hold40/labels/val... 48 images, 5 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 48/48 2.1Kit/s 0.0s
val: New cache created: /content/RPS_TestSet_hold40/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 48/48 33.9it/s 1.4s
               

In [ ]:
import os
TESTS = [('강사님 22장 (쉬움)', '/content/RPS_Dataset_YOLO_v6/data.yaml'),
         ('hold40 48장',        '/content/RPS_TestSet_hold40/data.yaml')]
MODELS = [('v1', S+'v1_best.tflite'), ('v3', S+'v3/best.tflite'),
          ('v4', S+'v4/best.tflite'), ('v6', S+'v6/best.tflite')]

res = {}
for tn, td in TESTS:
    for mn, mp in MODELS:
        if not os.path.exists(mp): continue
        r = YOLO(mp).val(data=td, imgsz=320, split='val', verbose=False, plots=False)
        res[(tn, mn)] = (r.box.map50, r.box.map)

print()
print('%-22s %-18s %-18s %-18s %-18s' % ('시험지', 'v1', 'v3', 'v4', 'v6'))
print('-' * 98)
for tn, _ in TESTS:
    cells = []
    for mn, _ in MODELS:
        v = res.get((tn, mn))
        cells.append('%.3f / %.3f' % v if v else '-')
    print('%-22s %-18s %-18s %-18s %-18s' % tuple([tn] + cells))
print()
print('칸 안의 값은  mAP50 / mAP50-95')

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/v1_best.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1958.8±841.5 MB/s, size: 69.5 KB)
val: Scanning /content/RPS_Dataset_YOLO_v6/labels/test.cache... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 4.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 33.4it/s 0.7s
                   all         22         32      0.983          1      0.995       0.81
Speed: 0.3ms preprocess, 24.3ms inference, 0.0ms loss, 0.9ms postprocess per image
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. 

In [ ]:
import pandas as pd, glob
for d in sorted(glob.glob('/content/drive/MyDrive/files/save/rps_yolo11n*/')):
    try:
        df = pd.read_csv(d + 'results.csv'); df.columns = df.columns.str.strip()
        i = df['metrics/mAP50(B)'].idxmax()
        print('%-24s 에포크 %d개  최고 mAP50 %.3f (%d번째)'
              % (d.split('/')[-2], len(df), df['metrics/mAP50(B)'][i], df['epoch'][i]))
    except Exception as e:
        print(d.split('/')[-2], '읽기 실패:', e)

rps_yolo11n-2            에포크 80개  최고 mAP50 0.995 (29번째)
rps_yolo11n-4            에포크 80개  최고 mAP50 0.995 (30번째)
rps_yolo11n              에포크 2개  최고 mAP50 0.700 (2번째)


In [ ]:
import os
from ultralytics import YOLO
S = '/content/drive/MyDrive/files/save/'
TESTS = [('강사님 22장', '/content/RPS_Dataset_YOLO_v6/data.yaml'),
         ('hold40 48장', '/content/RPS_TestSet_hold40/data.yaml')]
CHK = [('rps_yolo11n   (2에포크)',  S + 'rps_yolo11n/weights/best.tflite'),
       ('rps_yolo11n-2 (80에포크)', S + 'rps_yolo11n-2/weights/best.tflite'),
       ('rps_yolo11n-4 (80에포크)', S + 'rps_yolo11n-4/weights/best.tflite'),
       ('v4/ 에 들어있는 것',       S + 'v4/best.tflite')]

for tn, td in TESTS:
    print('===', tn, '===')
    for n, p in CHK:
        if not os.path.exists(p):
            print('  없음:', n); continue
        r = YOLO(p).val(data=td, imgsz=320, verbose=False, plots=False)
        print('  %-26s mAP50 %.3f  mAP50-95 %.3f' % (n, r.box.map50, r.box.map))
    print()

=== 강사님 22장 ===
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/rps_yolo11n/weights/best.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1132.7±509.3 MB/s, size: 68.0 KB)
val: Scanning /content/RPS_Dataset_YOLO_v6/labels/test.cache... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 5.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 33.0it/s 0.7s
                   all         22         32      0.922      0.252      0.632      0.271
Speed: 0.4ms preprocess, 23.3ms inference, 0.0ms loss, 2.1ms postprocess per image
  rps_yolo11n   (2에포크)       mAP50 0.632  mA

In [ ]:
import io, glob, os, time
S = '/content/drive/MyDrive/files/save/'
for d in sorted(glob.glob(S + 'rps_yolo11n*/')):
    a = d + 'args.yaml'
    if not os.path.exists(a):
        print(d, 'args.yaml 없음'); continue
    data = name = ''
    for ln in io.open(a, encoding='utf-8'):
        if ln.startswith('data:'): data = ln.split(':', 1)[1].strip()
        if ln.startswith('name:'): name = ln.split(':', 1)[1].strip()
    print('%-18s  데이터=%-42s  만든시각=%s'
          % (os.path.basename(d.rstrip('/')), data,
             time.strftime('%m-%d %H:%M', time.localtime(os.path.getmtime(a)))))

rps_yolo11n-2       데이터=/content/RPS_Dataset_YOLO_v2/data.yaml      만든시각=09-17 07:33
rps_yolo11n-4       데이터=/content/RPS_Dataset_YOLO_v3/data.yaml      만든시각=09-17 07:43
rps_yolo11n         데이터=/content/RPS_Dataset_YOLO_v4/data.yaml      만든시각=09-17 10:15


In [ ]:
!ls -d {save_dir}v*/ && echo '---' && ls -lh {save_dir}v7/

/content/drive/MyDrive/files/save/v3/  /content/drive/MyDrive/files/save/v6/
/content/drive/MyDrive/files/save/v4/  /content/drive/MyDrive/files/save/v7/
---
total 16M
-rw------- 1 root root 2.9M Sep 18 03:39 best_int8.tflite
-rw------- 1 root root  11M Sep 18 03:39 best.tflite
-rw------- 1 root root 2.8M Sep 18 03:39 best_w8a32.tflite


In [ ]:
!ls -lh {save_dir}v9/

total 11M
-rw------- 1 root root 11M Sep 18 04:11 best.tflite


In [ ]:
import os
from ultralytics import YOLO

!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_today.zip -d /content
TODAY = '/content/RPS_TestSet_today/data.yaml'
EASY  = '/content/RPS_Dataset_YOLO_v9/data.yaml'
S = '/content/drive/MyDrive/files/save/'

MODELS = [
    ('v1   104장', S + 'v1_best.tflite'),
    ('v2   410장', S + 'rps_yolo11n-2/weights/best.tflite'),
    ('v3   724장', S + 'v3/best.tflite'),
    ('v9 1,318장', S + 'v9/best.tflite'),
]

res = {}
for name, p in MODELS:
    if not os.path.exists(p):
        print('없음:', p); continue
    for tag, data in (('easy', EASY), ('today', TODAY)):
        r = YOLO(p).val(data=data, imgsz=320, verbose=False, plots=False)
        res[(name, tag)] = (r.box.map50, r.box.map, r.box.mp, r.box.mr)

print()
print('%-12s %-18s %-18s' % ('모델', '교수님 22장', '오늘 233장 (공정)'))
print('-' * 52)
for name, _ in MODELS:
    f = lambda v: '%.3f / %.3f' % (v[0], v[1]) if v else '-'
    print('%-12s %-18s %-18s' % (name, f(res.get((name,'easy'))), f(res.get((name,'today')))))
print('\n칸 = mAP50 / mAP50-95\n')
for name, _ in MODELS:
    b = res.get((name, 'today'))
    if b: print('%-12s 오늘 시험지  정밀도 %.3f  재현율 %.3f' % (name, b[2], b[3]))

WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/v1_best.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1278.0±694.9 MB/s, size: 69.9 KB)
val: Scanning /content/RPS_Dataset_YOLO_v9/labels/test.cache... 22 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 22/22 3.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 22/22 33.0it/s 0.7s
                   all         22         32      0.983          1      0.995       0.81
Speed: 0.3ms preprocess, 24.4ms inference, 0.0ms loss, 1.0ms postprocess per image
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. 

In [ ]:
!grep '^data:' {save_dir}rps_yolo11n-2/args.yaml

data: /content/RPS_Dataset_YOLO_v7/data.yaml


In [4]:
import os
from ultralytics import YOLO
save_dir = '/content/drive/MyDrive/files/save/'

for tag in ('A', 'B'):
    z = f'/content/drive/MyDrive/files/RPS_Dataset_YOLO_{tag}.zip'
    !unzip -q -o {z} -d /content
    root = f'/content/RPS_Dataset_YOLO_{tag}'
    data = root + '/data.yaml'
    run  = f'/content/rps_runs/ab_{tag}'
    print('=' * 60); print(tag, '학습 시작 (GPU, 80 epoch)'); print('=' * 60)
    !rm -rf {run}
    !yolo detect train model=yolo11n.pt data={data} \
      imgsz=320 epochs=80 batch=32 device=0 workers=4 \
      project=/content/rps_runs degrees=15.0 flipud=0.5 \
      fliplr=0.5 mosaic=1.0 name=ab_{tag}
    ep = sum(1 for _ in open(run + '/results.csv')) - 1
    if ep < 80:
        raise RuntimeError('%s 학습이 %d epoch에서 끊겼다' % (tag, ep))
    !mkdir -p {save_dir}{tag}
    !cp -r {run} {save_dir}
    !yolo export model={run}/weights/best.pt format=litert imgsz=320 data={data}
    !cp {run}/weights/*.tflite {save_dir}{tag}/

!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_hold40.zip -d /content
print()
print('%-22s %7s %9s %7s %7s' % ('', 'mAP50', 'mAP50-95', '정밀도', '재현율'))
print('-' * 56)
for tag, label in (('A', 'A  팀원 라벨'), ('B', 'B  내가 다시 라벨')):
    p = f'{save_dir}{tag}/best.tflite'
    if not os.path.exists(p):
        print('없음:', p); continue
    r = YOLO(p).val(data='/content/RPS_TestSet_hold40/data.yaml', imgsz=320,
                    verbose=False, plots=False)
    print('%-22s %7.3f %9.3f %7.3f %7.3f'
          % (label, r.box.map50, r.box.map, r.box.mp, r.box.mr))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
A 학습 시작 (GPU, 80 epoch)
New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO_A/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=No

In [5]:
import os
from ultralytics import YOLO
S = '/content/drive/MyDrive/files/save/'
!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_today.zip -d /content
print()
print('%-22s %7s %9s %7s %7s' % ('', 'mAP50', 'mAP50-95', '정밀도', '재현율'))
print('-' * 56)
for tag, label in (('A', 'A  팀원 라벨'), ('B', 'B  내가 다시 라벨')):
    r = YOLO(S + tag + '/best.tflite').val(
        data='/content/RPS_TestSet_today/data.yaml', imgsz=320,
        verbose=False, plots=False)
    a = r.box.maps
    print('%-22s %7.3f %9.3f %7.3f %7.3f   가위 %.3f 주먹 %.3f 보 %.3f'
          % (label, r.box.map50, r.box.map, r.box.mp, r.box.mr, a[0], a[1], a[2]))


                         mAP50  mAP50-95     정밀도     재현율
--------------------------------------------------------
WARNING ⚠️ Unable to automatically guess model task, assuming 'task=detect'. Explicitly define task for your model, i.e. 'task=detect', 'segment', 'classify', 'pose', 'obb' or 'semantic'.
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Loading /content/drive/MyDrive/files/save/A/best.tflite for LiteRT inference...
Setting batch=1 input of shape (1, 3, 320, 320)
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1483.9±441.4 MB/s, size: 52.1 KB)
val: Scanning /content/RPS_TestSet_today/labels/val... 233 images, 25 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 233/233 1.9Kit/s 0.1s
val: New cache created: /content/RPS_TestSet_today/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 233/233 30.3it/s 7.7s
                   all        233        328      0.998      0.997    

In [6]:
import os
from ultralytics import YOLO
save_dir = '/content/drive/MyDrive/files/save/'
tag  = 'v10'
z    = f'/content/drive/MyDrive/files/RPS_Dataset_YOLO_{tag}.zip'
root = f'/content/RPS_Dataset_YOLO_{tag}'
data = root + '/data.yaml'
run  = f'/content/rps_runs/{tag}'

!unzip -q -o {z} -d /content
!rm -rf {run}
!yolo detect train model=yolo11n.pt data={data} \
  imgsz=320 epochs=80 batch=32 device=0 workers=4 \
  project=/content/rps_runs degrees=15.0 flipud=0.5 \
  fliplr=0.5 mosaic=1.0 name={tag}

ep = sum(1 for _ in open(run + '/results.csv')) - 1
if ep < 80: raise RuntimeError('%d epoch에서 끊겼다' % ep)
!mkdir -p {save_dir}{tag}
!cp -r {run} {save_dir}
!yolo export model={run}/weights/best.pt format=litert imgsz=320 data={data}
!cp {run}/weights/*.tflite {save_dir}{tag}/

# ── 채점 : v3 / v9 / v10 을 같은 시험지로 ──────────────
!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_today.zip -d /content
TODAY = '/content/RPS_TestSet_today/data.yaml'
S = save_dir
print()
print('%-24s %7s %9s %7s %7s' % ('', 'mAP50', 'mAP50-95', '정밀도', '재현율'))
print('-' * 58)
for n, p in [('v3   724장', S+'v3/best.tflite'),
             ('v10  884장 (v4a 없음)', S+'v10/best.tflite'),
             ('v9  1,318장 (v4a 포함)', S+'v9/best.tflite')]:
    if not os.path.exists(p): print('없음:', p); continue
    r = YOLO(p).val(data=TODAY, imgsz=320, verbose=False, plots=False)
    print('%-24s %7.3f %9.3f %7.3f %7.3f'
          % (n, r.box.map50, r.box.map, r.box.mp, r.box.mr))

New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO_v10/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosai

In [7]:
import os
from ultralytics import YOLO
save_dir = '/content/drive/MyDrive/files/save/'
DATA = '/content/RPS_Dataset_YOLO_v10/data.yaml'
!unzip -q -o /content/drive/MyDrive/files/RPS_Dataset_YOLO_v10.zip -d /content
!unzip -q -o /content/drive/MyDrive/files/RPS_TestSet_today.zip -d /content
TODAY = '/content/RPS_TestSet_today/data.yaml'

for sd in (1, 2):
    run = f'/content/rps_runs/v10_s{sd}'
    print('=' * 60); print('seed', sd); print('=' * 60)
    !rm -rf {run}
    !yolo detect train model=yolo11n.pt data={DATA} \
      imgsz=320 epochs=80 batch=32 device=0 workers=4 seed={sd} \
      project=/content/rps_runs degrees=15.0 flipud=0.5 \
      fliplr=0.5 mosaic=1.0 name=v10_s{sd}
    ep = sum(1 for _ in open(run + '/results.csv')) - 1
    if ep < 80: raise RuntimeError('%d epoch에서 끊겼다' % ep)
    !yolo export model={run}/weights/best.pt format=litert imgsz=320 data={DATA}
    !mkdir -p {save_dir}v10_s{sd}
    !cp {run}/weights/*.tflite {save_dir}v10_s{sd}/

print()
print('%-22s %7s %9s' % ('', 'mAP50', 'mAP50-95'))
print('-' * 42)
for n, p in [('v3   (참고)',      save_dir+'v3/best.tflite'),
             ('v10  seed 0',      save_dir+'v10/best.tflite'),
             ('v10  seed 1',      save_dir+'v10_s1/best.tflite'),
             ('v10  seed 2',      save_dir+'v10_s2/best.tflite')]:
    if not os.path.exists(p): print('없음:', p); continue
    r = YOLO(p).val(data=TODAY, imgsz=320, verbose=False, plots=False)
    print('%-22s %7.3f %9.3f' % (n, r.box.map50, r.box.map))

seed 1
New https://pypi.org/project/ultralytics/8.4.155 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.98 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/RPS_Dataset_YOLO_v10/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dis=6.0, distill_model=None, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=80, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937